# ARC-AGI-3 — Hybrid Explorer Agent

General, training-free interactive agent (MIT-0). Motion model (avatar + per-action
displacement) + coordinate navigation to candidate goals, with graph-based
exploration/exploitation fallback. Runs through the official ARC-AGI-3-Agents framework
against the gateway-served games.

**Setup:** attach the `arcagi3-agent` dataset (this repo's `src/`) to the notebook.


In [ ]:
# Install the ARC-AGI-3 toolkit + engine offline from the competition wheels.
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv


In [ ]:
%%writefile /kaggle/working/my_agent.py
# =====================================================================
# ARC-AGI-3 submission agent (ARC Prize 2026)
# A general, training-free interactive agent: motion model (avatar + per-action
# displacement) + coordinate navigation to candidate goals, with graph-based
# exploration/exploitation fallback. Reactive one-action-per-call interface.
#
# The actual logic lives in the `arcagi3` package, shipped as a Kaggle dataset and
# added to sys.path below. This file is a thin adapter to the official Agent base.
# =====================================================================
import os
import sys
import time

# Make the arcagi3 package importable. The package is uploaded as a Kaggle dataset;
# try the common attach paths plus the working dir.
_CANDIDATES = [
    "/kaggle/input/arcagi3-agent",
    "/kaggle/input/arcagi3-agent/src",
    "/kaggle/input/arcagi3",
    "/kaggle/working/arcagi3-agent",
    "/kaggle/working/ARC-AGI-3-Agents/agents/templates",
    os.path.dirname(os.path.abspath(__file__)),
    os.path.join(os.path.dirname(os.path.abspath(__file__)), "src"),
]
for _p in _CANDIDATES:
    if _p and os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

from arcagi3.policy import HybridPolicy  # noqa: E402

try:  # the official framework base; stubbed for local dev/testing
    from agents.agent import Agent as _BaseAgent  # noqa: E402
except Exception:  # pragma: no cover
    class _BaseAgent:  # minimal stub
        def __init__(self, *a, **k):
            self.game_id = k.get("game_id", "?")
            self.action_counter = 0
            self.frames = []

from arcengine import GameAction as _GA  # noqa: E402
from arcengine import GameState as _GS  # noqa: E402

# Stop ~5 min before the 8h soft budget (Kaggle hard limit is 12h overall).
_TIME_BUDGET_S = 8 * 3600 - 5 * 60


class MyAgent(_BaseAgent):
    """Hybrid explorer adapted to the official reactive Agent interface."""

    MAX_ACTIONS = float("inf")

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._pol = HybridPolicy()
        self._t0 = time.time()

    def is_done(self, frames, latest_frame):
        try:
            if latest_frame.state is _GS.WIN:
                return True
        except Exception:
            pass
        return (time.time() - self._t0) >= _TIME_BUDGET_S

    def choose_action(self, frames, latest_frame):
        try:
            import numpy as np

            arr = np.asarray(latest_frame.frame, dtype=np.int8)
            grid = arr[-1] if arr.ndim == 3 else arr
            st = latest_frame.state
            avail = []
            for a in (getattr(latest_frame, "available_actions", None) or []):
                avail.append(a.value if hasattr(a, "value") else int(a))
            token = self._pol.decide(
                grid,
                gstate_terminal=(st is _GS.GAME_OVER),
                gstate_notplayed=(st is _GS.NOT_PLAYED),
                levels=int(getattr(latest_frame, "levels_completed", 0) or 0),
                available=avail,
            )
            if token[0] == "reset":
                act = _GA.RESET
                act.reasoning = "reset"
                return act
            if token[0] == "S":
                act = _GA.from_id(token[1])
                act.reasoning = "hybrid:explore/navigate"
                return act
            act = _GA.ACTION6
            act.set_data({"x": int(token[1]), "y": int(token[2])})
            act.reasoning = "hybrid:click"
            return act
        except Exception as e:  # never die mid-game
            act = _GA.ACTION1
            try:
                act.reasoning = f"fallback: {e}"
            except Exception:
                pass
            return act


In [ ]:
import os, shutil, glob

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # 1) Wait for the gateway that serves the private games.
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # 2) Copy the official agents framework to a writable location.
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # 3) Drop our agent + the arcagi3 package next to the framework templates so it
    #    imports cleanly even if the dataset attach path differs.
    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py
    for cand in ['/kaggle/input/arcagi3-agent/src/arcagi3',
                 '/kaggle/input/arcagi3-agent/arcagi3',
                 '/kaggle/input/arcagi3/arcagi3']:
        if os.path.isdir(cand):
            dst = '/kaggle/working/ARC-AGI-3-Agents/agents/templates/arcagi3'
            if not os.path.isdir(dst):
                shutil.copytree(cand, dst)
            break

    # 4) Minimal __init__.py: register only what we need (avoid heavy template imports).
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write('''from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
}
''')

    # 5) .env pointing the framework at the gateway (ONLINE mode, no local env files).
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write('''SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
''')

    # 6) Play all games. The gateway records the scorecard -> submission.
    !cd /kaggle/working/ARC-AGI-3-Agents && MPLBACKEND=agg python main.py --agent myagent


In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Local/commit run: emit a placeholder submission so the notebook saves cleanly.
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 0]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    submission.head()
